# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [24]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [25]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print("Total revenue:", total_revenue)
print("Total units:", total_units)

Total revenue: 8520.0
Total units: 783


The total revenue is 8520. 783 units are sold in 400 orders.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [26]:
# TODO
category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
by_category= pd.DataFrame({'revenue': category,
    'share_percent': category /total_revenue * 100})
by_category

,revenue,share_percent
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


Food has the most revenue with 4293 and about 50.39% of the total.
Merch has 1771.5 in revenue and about 20.79% of the total.
Drink has 1554 in revenue and about 18.24% of the total.
RainGear has 901.5 in revenue and about 10.58% of the total.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [27]:
# TODO
vendor = df.groupby('vendor_id')['revenue'].agg(['mean', 'count'])
vendor

,mean,count
vendor_id,,
V-01,22.595745,94
V-05,20.580645,93
V-10,20.314286,105
V-18,21.750000,108


V-01 has the highest average order revenue at about 22.60 in 94 orders. V-05 has an average order revenue of about 20.58 in 93 orders.
V-10 has an average order revenue of 20.31 in 105 orders.
V-18 has an average order revenue of 21.75 in 108 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [28]:
# TODO
merch_share = category['Merch'] / total_revenue * 100
print(round(merch_share,1))

20.8


Merch makes up about 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [29]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
print(len(df), len(joined))
print(df['revenue'].sum(), joined['revenue'].sum())
joined[joined['vendor_name'].isna()]['vendor_id'].unique()


400 400
8520.0 8520.0


array(['V-18'], dtype=object)

**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18. I kept it and left its vendor name missing because its orders are still valid.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [32]:
# TODO
pivot = pd.pivot_table(df, values='revenue',
                       index='vendor_id',
                       columns='category',
                       aggfunc='sum',
                       margins=True)

pivot

category,Drink,Food,Merch,RainGear,All
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


From the table, V-18 has the most total revenue with 2349. Food has the most revenue with 4293. The total revenue is 8520.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [33]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

Vendors should plan what to stock for the next game based on sales and revenue from this game. Food made 4293 dollars, which is 50.39% of total revenue. However, RainGear only made 901.5. As a result, they could prepare more Food and less RainGear next game based on the demand from this game.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

I think Q5 is the least trustworthy because V-18 does not have a matching vendor name. Its orders are still in the data, but the vendor name is missing.
